# SkyFlow: Machine Learning Foundations for Air Traffic Congestion & Delay Forecasting
## Comprehensive Academic Laboratory Mapping Course Outcomes (CO1 to CO6)

**Author:** Academic Machine Learning Research Team  
**Domain:** Aviation Operations, Delay Propagation, and Airport Congestion Modeling  
**Course Outcomes Covered:**
- **CO1**: Model Selection, Cross-Validation, Regularization, and Hyperparameter Tuning
- **CO2**: Advanced Linear Regression, SLR, MLR, Polynomial Regression, Diagnostics & Regularization (Ridge/Lasso)
- **CO3**: Support Vector Machines (SVM) - 2D/3D Hyperplanes, Maximal & Soft Margins, Slack Variables ($\xi$), Cost ($C$)
- **CO4**: Kernel Methods, Feature Transformations, The Kernel Trick (Linear, Poly, RBF), Support Vector Regression (SVR)
- **CO5**: Decision Trees, Impurity Measures (Gini vs Entropy), Information Gain, Pruning, Feature Importance (MDI)
- **CO6**: Ensemble Techniques, Random Forests, Out-of-Bag (OOB) Error Comprehension, Feature Importances

--- 
## 0. Environment Setup & Data Pipeline
We load the multi-airport hourly operational dataset consisting of 17,280 continuous hourly records across major hubs (HYD, DEL, BOM, SFO, ATL, ORD, LAX, DFW).

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats

# Ensure local src modules are discoverable
BASE_DIR = os.getcwd()
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

from src.data_loader import load_air_traffic_data, get_regression_data, get_classification_data, FEATURE_COLS

df = load_air_traffic_data()
print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

--- 
## 1. Unit I: Model Selection and Evaluation (CO1)
### Course Outcome 1:
> *Analyze the effectiveness of machine learning models using cross-validation for model selection, regularization, and hyperparameter tuning.*

### 1.1 The Bias-Variance Tradeoff
The expected generalization error of a model decomposes into:
$$\mathbb{E}[(y - \hat{f}(x))^2] = \text{Bias}[\hat{f}(x)]^2 + \text{Var}[\hat{f}(x)] + \sigma^2$$
- **Underfitting (High Bias)**: The model is too simple to capture the underlying pattern.
- **Overfitting (High Variance)**: The model fits the training noise rather than the signal.

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

reg_data = get_regression_data(df)
X_train, X_test = reg_data["X_train"], reg_data["X_test"]
y_train, y_test = reg_data["y_train"], reg_data["y_test"]

depths = range(1, 21)
train_errors, test_errors = [], []

for d in depths:
    dt = DecisionTreeRegressor(max_depth=d, random_state=42)
    dt.fit(X_train, y_train)
    train_errors.append(mean_squared_error(y_train, dt.predict(X_train)))
    test_errors.append(mean_squared_error(y_test, dt.predict(X_test)))

best_depth = depths[np.argmin(test_errors)]
print(f"Optimal Tree Depth minimizing Test MSE: {best_depth}")

plt.figure(figsize=(9, 4.5))
plt.plot(depths, train_errors, label='Train MSE (Bias)', marker='o', color='#2563eb')
plt.plot(depths, test_errors, label='Test MSE (Variance)', marker='s', color='#dc2626')
plt.axvline(best_depth, color='#16a34a', linestyle='--', label=f'Optimal Tradeoff (depth={best_depth})')
plt.title('Empirical Bias-Variance Tradeoff on Airport Delay Data')
plt.xlabel('Model Complexity (Tree Depth)')
plt.ylabel('Mean Squared Error')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 1.2 K-Fold Cross Validation & Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import KFold, cross_val_score, GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeClassifier

# 5-Fold Cross-Validation on Ridge Regression
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
r2_scores = cross_val_score(Ridge(alpha=1.0), pd.concat([X_train, X_test]), pd.concat([y_train, y_test]), cv=kfold, scoring='r2')
print(f"5-Fold Cross Validation R2 Scores: {np.round(r2_scores, 4)}")
print(f"Mean R2: {r2_scores.mean():.4f} (+/- {r2_scores.std():.4f})")

# Hyperparameter Tuning via GridSearchCV
clf_data = get_classification_data(df)
param_grid = {'max_depth': [3, 5, 8], 'min_samples_leaf': [1, 5, 10]}
grid = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=3, scoring='accuracy', n_jobs=1)
grid.fit(clf_data["X_train"], clf_data["y_train"])
print(f"GridSearchCV Best Parameters: {grid.best_params_}")
print(f"GridSearchCV Best Accuracy: {grid.best_score_ * 100:.2f}%")

--- 
## 2. Unit II: Advanced Regression Techniques (CO2)
### Course Outcome 2:
> *Analyze different linear regression techniques, including simple linear regression, multiple linear regression, polynomial regression, and regularization, to model and make predictions on datasets.*

### 2.1 Simple Linear Regression (Closed-Form vs Matrix Formula)
Formula:
$$\hat{\beta}_1 = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sum (x_i - \bar{x})^2}, \quad \hat{\beta}_0 = \bar{y} - \hat{\beta}_1 \bar{x}$$
Matrix Formulation:
$$\hat{\boldsymbol{\beta}} = (X^T X)^{-1} X^T \mathbf{y}$$

In [ ]:
x = df["runway_utilization"].values
y = df["avg_departure_delay"].values

# Scalar Manual Estimators
x_bar, y_bar = np.mean(x), np.mean(y)
beta_1_manual = np.sum((x - x_bar) * (y - y_bar)) / np.sum((x - x_bar)**2)
beta_0_manual = y_bar - beta_1_manual * x_bar

# Matrix Formulation
X_mat = np.column_stack([np.ones_like(x), x])
beta_mat = np.linalg.inv(X_mat.T @ X_mat) @ (X_mat.T @ y)

print(f"Manual Closed-Form: Intercept = {beta_0_manual:.4f}, Slope = {beta_1_manual:.4f}")
print(f"Matrix (X^T X)^(-1) X^T y: Intercept = {beta_mat[0]:.4f}, Slope = {beta_mat[1]:.4f}")

### 2.2 Multiple Linear Regression & Assumption Diagnostics (VIF, Residuals, Normality)

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso

mlr = LinearRegression()
mlr.fit(X_train, y_train)
preds = mlr.predict(X_test)
residuals = y_test - preds

print(f"MLR Test Performance: R2 = {mlr.score(X_test, y_test):.4f}")

# Variance Inflation Factor (VIF)
vifs = []
for col in X_train.columns:
    other = [c for c in X_train.columns if c != col]
    r2_j = LinearRegression().fit(X_train[other], X_train[col]).score(X_train[other], X_train[col])
    vifs.append({'Feature': col, 'VIF': round(1.0 / (1.0 - r2_j), 2) if r2_j < 1 else np.inf})
print(pd.DataFrame(vifs))

--- 
## 3. Unit III & IV: Support Vector Machines & The Kernel Trick (CO3 & CO4)
### Course Outcomes 3 & 4:
> *CO3: Apply SVM techniques, maximal margin classifier, soft margin classifier, slack variables, and cost C.*  
> *CO4: Analyze the role of kernel methods and the kernel trick (Linear, Polynomial, RBF) in Python.*

In [ ]:
from sklearn.svm import SVC, SVR

X_clf_tr = clf_data["X_train"]
y_clf_tr = clf_data["y_train"]
X_clf_te = clf_data["X_test"]
y_clf_te = clf_data["y_test"]

for kernel in ['linear', 'poly', 'rbf']:
    svm = SVC(kernel=kernel, C=1.0, random_state=42)
    svm.fit(X_clf_tr, y_clf_tr)
    acc = svm.score(X_clf_te, y_clf_te)
    print(f"SVM Kernel: {kernel.upper():<7} | Accuracy: {acc * 100:.2f}% | Support Vectors: {len(svm.support_vectors_)}")

# Support Vector Regression (SVR)
svr = SVR(kernel='rbf', C=10.0, epsilon=0.2)
svr.fit(X_train, y_train)
print(f"SVR Test R2 Score: {svr.score(X_test, y_test):.4f}")

--- 
## 4. Unit V: Decision Trees, Impurity Measures & Pruning (CO5)
### Course Outcome 5:
> *Explore decision tree techniques, including measuring impurity (Gini vs Entropy), feature importance, and hyperparameters.*

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Scratch Calculation of Gini vs Entropy
counts = np.array([50, 30, 20])
probs = counts / counts.sum()
gini = 1.0 - np.sum(probs**2)
entropy = - np.sum([p * np.log2(p) for p in probs if p > 0])
print(f"Manual Gini Impurity: {gini:.4f}")
print(f"Manual Shannon Entropy: {entropy:.4f} bits")

# Trained Decision Tree Classifier & Feature Importances (MDI)
dt = DecisionTreeClassifier(max_depth=5, min_samples_leaf=10, random_state=42)
dt.fit(X_clf_tr, y_clf_tr)
imp_df = pd.DataFrame({
    'Feature': X_clf_tr.columns,
    'Importance': np.round(dt.feature_importances_, 4)
}).sort_values('Importance', ascending=False)
print("\nDecision Tree Feature Importances (Mean Decrease in Impurity):")
print(imp_df)

--- 
## 5. Unit VI: Ensemble Methods & Random Forests (CO6)
### Course Outcome 6:
> *Use ensemble techniques, including random forests and Out-of-Bag (OOB) error estimation, to improve model performance.*

In [ ]:
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

# Out-of-Bag (OOB) Score Demonstration
rf = RandomForestClassifier(n_estimators=100, max_features='sqrt', oob_score=True, random_state=42, n_jobs=1)
rf.fit(X_clf_tr, y_clf_tr)

print(f"Random Forest Out-of-Bag (OOB) Score: {rf.oob_score_ * 100:.2f}%")
print(f"Random Forest Test Set Accuracy:        {rf.score(X_clf_te, y_clf_te) * 100:.2f}%")

# Random Forest Regressor
rf_reg = RandomForestRegressor(n_estimators=100, max_features='sqrt', random_state=42, n_jobs=1)
rf_reg.fit(X_train, y_train)
print(f"Random Forest Regressor Test R2 Score:  {rf_reg.score(X_test, y_test):.4f}")

--- 
## 6. Synthesis & Master Benchmark Summary
We summarize all evaluated algorithms across both prediction targets.

In [ ]:
import json
with open(os.path.join(BASE_DIR, 'models', 'master_metrics.json'), 'r') as f:
    metrics = json.load(f)

print("=== REGRESSION BENCHMARK SUMMARY (FLIGHT DELAY) ===")
print(pd.DataFrame(metrics['regression_benchmark']).T)

print("\n=== CLASSIFICATION BENCHMARK SUMMARY (CONGESTION LEVEL) ===")
print(pd.DataFrame(metrics['classification_benchmark']).T)